### **FIMbench - `processing_floodmap`**

Standardize a **raw benchmark flood map** into the FIMbench database format.
Each source has its own processor class; each writes a renamed GeoTIFF,
`metadata.json`, and `AOI.gpkg` into a per-map folder under the destination.
Intersected HUC8 watershed info (US) is resolved on the fly from the ArcGIS
REST service.

| Class | Source |
| --- | --- |
| `Tier1Processor` | Aerial Imagery (AI) |
| `Tier2Processor` | Planet Scope Scene (PSS) |
| `Tier3Processor` | Sentinel 1A (S1A) |
| `FemaBleProcessor` | FEMA Base Level Engineering (BLE), return-period event |
| `HwmProcessor` | High Water Mark (HWM), single TIF + start/end date |

### **Install**

`fimbench` is deployed in the PyPI, install it.

In [ ]:
!uv pip install fimbench

### **Initialization**

In [ ]:
#`input_path` can be a **folder** (every `.tif` under it) or a **single** `.tif`.

input_folder = 'path/to/rasters'                  # a folder of .tif files
input_tif = 'path/to/Binary_20160103_BM.tif'     # or a single raster
out_dir = 'path/to/standardized-out'             # per-map folders are written here

### **Tier 1 - Aerial Imagery (AI)**

Every constructor keyword is **optional**; `None` keeps the class default.
Pass the optional keywords to override the metadata/encoding defaults.

In [ ]:
from fimbench import Tier1Processor

processor = Tier1Processor(
    sensor_code=None,    # short source code in filenames (default 'AI')
    source=None,         # creator name / email recorded in metadata.json
    nodata_val=None,     # raster nodata value (default -9999)
    block_size=None,     # GeoTIFF internal tile size (default 512)
    simplify_tol=None,   # AOI simplify tolerance in deg to generate the FIM extent geometry (default 0.0001 ~ 11 m)
)

processor.process(
    input_folder,             # input: a folder of .tif (or a single .tif)
    out_dir,                  # destination root for the standardized folder(s)
    flood_date=None,          # 'YYYYMMDD'; inferred per file for a folder, pass it for a single raster
    intermediate_folder=None, # scratch dir for temp rasters; None -> a temp dir is used and removed
)

### T**ier 2 (Planet) and Tier 3 (Sentinel-1A)**

In [ ]:
from fimbench import Tier2Processor, Tier3Processor

Tier2Processor(
    sensor_code=None,    # default 'PSS'
    source=None,         # creator name / email
    nodata_val=None,     # default -9999
    block_size=None,     # default 512
    simplify_tol=None,   # default 0.0001 deg
).process(
    input_folder,             # folder or single .tif
    out_dir,                  # destination root
    flood_date=None,          # 'YYYYMMDD' (inferred per file for a folder)
    intermediate_folder=None, # None -> temp dir
)

# Tier3Processor(...).process(...) takes the exact same arguments (default sensor_code 'S1A').

### **FEMA BLE- return-period event**

Like the tiers, but `process` takes an `event` (return period) instead of a date.

In [ ]:
from fimbench import FemaBleProcessor

FemaBleProcessor(
    sensor_code=None,    # default 'BLE'
    full_form=None,      # long-form source label for metadata
    source=None,         # creator / provider (default NOAA/NWS OWP)
    nodata_val=None,     # default -9999
    block_size=None,     # default 512
    simplify_tol=None,   # default 0.0001 deg
).process(
    input_folder,             # folder or single .tif
    out_dir,                  # destination root
    event='100',              # return period, e.g. '100' (100-yr), '500', ...
    intermediate_folder=None, # None -> temp dir
)

### **HWM- High Water Mark**

Single TIF and a start/end date range for the event.

In [ ]:
from fimbench import HwmProcessor

HwmProcessor(
    sensor_code=None,    # default 'HWM'
    full_form=None,      # long-form source label for metadata
    source=None,         # creator name / email
    nodata_val=None,     # default -9999
    block_size=None,     # default 512
    simplify_tol=None,   # default 0.0001 deg
).process(
    input_tif,                # input raster (or folder)
    out_dir,                  # destination root
    start_date='160928',      # event start, 'YYMMDD'
    end_date='161009',        # event end,   'YYMMDD'
    intermediate_folder=None, # None -> temp dir
)

### **Module-level shortcuts**

Each source module also exposes a `process(...)` shortcut that accepts the same
arguments plus `**overrides` (the constructor keywords above).

In [ ]:
from fimbench.processing_floodmap import tier1, fema_ble, hwm

tier1.process(
    input_folder,             # folder or single .tif
    out_dir,                  # destination root
    flood_date=None,          # 'YYYYMMDD' (inferred per file for a folder)
    intermediate_folder=None, # None -> temp dir
    # source='Dr. X, x@uni.edu',  # any constructor default via **overrides
)

fema_ble.process(input_folder, out_dir, event='100', intermediate_folder=None)
hwm.process(input_tif, out_dir, start_date='160928', end_date='161009', intermediate_folder=None)

---
The standardized folder (GeoTIFF + `metadata.json` + `AOI.gpkg`) is now ready to
be cataloged by `webcontent_utils` and pushed to S3 by `publish`.